# Práctica 1.1 — Probabilidad y Estadística
## Análisis de la evolución poblacional mundial

## Objetivo

Utilizar un conjunto de datos real para comprobar el funcionamiento del entorno de análisis de datos y aplicar medidas estadísticas descriptivas básicas utilizando Pandas y NumPy.

## Actividad 1 — Identificación de la fuente

### Propósito

Reconocer la procedencia del conjunto de datos antes de utilizarlo y documentar correctamente su fuente.

### Cita de la fuente

Los datos utilizados en esta práctica provienen de **Our World in Data (OWID)**.

La fuente original es **United Nations, World Population Prospects (2024)**.

**Cita:**

> UN, World Population Prospects (2024) – processed by Our World in Data. "Annual change in population – UN WPP" [dataset]. United Nations, "World Population Prospects"; United Nations, "World Population Prospects - Interim Update" [original data].

### Preguntas

**1. ¿Quién produjo originalmente los datos?**

Los datos fueron producidos originalmente por la **Organización de las Naciones Unidas (ONU)**, a través de su publicación *World Population Prospects (2024)*.

**2. ¿Qué organización los procesa para su utilización en Our World in Data?**

**Our World in Data (OWID)** es la organización que recopila, estandariza y procesa los datos originales de la ONU para publicarlos en su plataforma.

**3. ¿Qué representa la variable que analizarás?**

La variable analizada es `Annual change in population` (cambio anual de población): representa el cambio **neto** de población de una entidad, calculado como la diferencia entre la población al 1 de julio de dos años consecutivos. Este cambio neto combina simultáneamente nacimientos, muertes y migración.

## Actividad 2 — Carga y exploración de datos

### Propósito

Comprobar que el entorno de Python, Pandas y Jupyter funciona correctamente y realizar una primera exploración del dataset.

### 2.1 Importar las bibliotecas

In [1]:
import pandas as pd
import numpy as np

### 2.2 Cargar el archivo CSV

In [2]:
df = pd.read_csv("../data/raw/annual-population-growth/annual-population-growth.csv")
df.head()

,Entity,Code,Year,Annual change in population,Annual population change (Projected)
0,Afghanistan,AFG,1951,103163.0,NaN
1,Afghanistan,AFG,1952,108441.0,NaN
2,Afghanistan,AFG,1953,108919.0,NaN
3,Afghanistan,AFG,1954,111251.0,NaN
4,Afghanistan,AFG,1955,119027.0,NaN


**Nota:** el archivo descargado incluye tanto el histórico (1951–2023) como proyecciones hasta 2100 (columna `Annual population change (Projected)`). Como esta práctica utiliza **únicamente los datos históricos**, se filtran las filas donde la variable histórica sí tiene valor y se descarta la columna de proyección.

In [3]:
df = df[df["Annual change in population"].notna()].drop(columns=["Annual population change (Projected)"]).reset_index(drop=True)
df.head()

,Entity,Code,Year,Annual change in population
0,Afghanistan,AFG,1951,103163.0
1,Afghanistan,AFG,1952,108441.0
2,Afghanistan,AFG,1953,108919.0
3,Afghanistan,AFG,1954,111251.0
4,Afghanistan,AFG,1955,119027.0


### 2.3 Explorar el DataFrame

In [4]:
# Primeras filas
df.head()

,Entity,Code,Year,Annual change in population
0,Afghanistan,AFG,1951,103163.0
1,Afghanistan,AFG,1952,108441.0
2,Afghanistan,AFG,1953,108919.0
3,Afghanistan,AFG,1954,111251.0
4,Afghanistan,AFG,1955,119027.0


In [5]:
# Dimensiones del DataFrame
df.shape

(18688, 4)

In [6]:
# Nombres de columnas
df.columns.tolist()

['Entity', 'Code', 'Year', 'Annual change in population']

In [7]:
# Tipos de datos
df.dtypes

Entity                             str
Code                               str
Year                             int64
Annual change in population    float64
dtype: object

In [8]:
# Estadísticas descriptivas generales
df.describe()

,Year,Annual change in population
count,18688.000000,1.868800e+04
mean,1987.000000,2.063019e+06
std,21.071871,9.440859e+06
min,1951.000000,-3.315935e+06
25%,1969.000000,1.781000e+03
50%,1987.000000,4.689800e+04
75%,2005.000000,3.021128e+05
max,2023.000000,9.337137e+07


In [9]:
# Cantidad de valores faltantes por columna
df.isnull().sum()

Entity                           0
Code                           584
Year                             0
Annual change in population      0
dtype: int64

In [10]:
# Cantidad de entidades distintas
df["Entity"].nunique()

256

### Preguntas

**1. ¿Cuántas filas y columnas tiene el dataset?**

Una vez filtrado a datos históricos, el dataset tiene **18,688 filas y 4 columnas** (`Entity`, `Code`, `Year`, `Annual change in population`).

**2. ¿Qué tipo de datos contiene la variable `Year`?**

`Year` es de tipo **`int64`** (entero), ya que representa años completos.

**3. ¿Existen valores faltantes?**

Sí, pero solo en la columna `Code`, con **584 valores faltantes**. Esto ocurre porque `Code` almacena el código ISO de país, y muchas entidades del dataset son agregados (continentes, regiones, grupos de ingreso, "World", etc.) que no tienen un código de país asociado. No hay valores faltantes en `Entity`, `Year` ni en la variable de cambio poblacional.

**4. ¿Cuántas entidades diferentes contiene el dataset?**

El dataset contiene **256 entidades** distintas, entre países individuales y agregados regionales/mundiales.

## Actividad 3 — Estadística descriptiva

### Propósito

Aplicar medidas estadísticas básicas a una variable poblacional.

### Selección del país y del período

En lugar de comparar entidades en un único año, en esta actividad se analiza la **evolución temporal de un solo país**. Se elige un país cuyo nombre comience con la letra **"M"**: **México**. El período histórico analizado inicia en el año **2006** y llega hasta el último año histórico disponible (2023).

Se construye un **DataFrame nuevo** (`df_pais`) filtrado únicamente con las observaciones de México a partir de 2006, y a partir de este nuevo DataFrame se realiza el resto de la actividad.

In [11]:
anio_inicio = 2006
pais = "Mexico"

df_pais = df[(df["Entity"] == pais) & (df["Year"] >= anio_inicio)].reset_index(drop=True)
df_pais

,Entity,Code,Year,Annual change in population
0,Mexico,MEX,2006,1442162.0
1,Mexico,MEX,2007,1520690.0
2,Mexico,MEX,2008,1599930.0
3,Mexico,MEX,2009,1625434.0
4,Mexico,MEX,2010,1624178.0
5,Mexico,MEX,2011,1619605.0
6,Mexico,MEX,2012,1574705.0
7,Mexico,MEX,2013,1525364.0
8,Mexico,MEX,2014,1440693.0
9,Mexico,MEX,2015,1288045.0


Sobre la variable `Annual change in population` de México entre 2006 y 2023, se calculan las medidas estadísticas descriptivas:

In [12]:
col = "Annual change in population"

media = df_pais[col].mean()
mediana = df_pais[col].median()
varianza = df_pais[col].var()
desviacion_estandar = df_pais[col].std()
minimo = df_pais[col].min()
maximo = df_pais[col].max()

print(f"Media:               {media:,.2f}")
print(f"Mediana:              {mediana:,.2f}")
print(f"Varianza:             {varianza:,.2f}")
print(f"Desviación estándar:  {desviacion_estandar:,.2f}")
print(f"Mínimo:               {minimo:,.2f}")
print(f"Máximo:               {maximo:,.2f}")

Media:               1,329,347.61
Mediana:              1,364,369.00
Varianza:             63,748,206,949.78
Desviación estándar:  252,484.07
Mínimo:               849,098.00
Máximo:               1,625,434.00


In [13]:
anio_min = df_pais.loc[df_pais[col].idxmin(), "Year"]
anio_max = df_pais.loc[df_pais[col].idxmax(), "Year"]

print(f"Año con el valor mínimo ({minimo:,.0f}): {anio_min}")
print(f"Año con el valor máximo ({maximo:,.0f}): {anio_max}")

Año con el valor mínimo (849,098): 2021
Año con el valor máximo (1,625,434): 2009


### Preguntas

**1. ¿Cuál es el valor de la media?**

La media del cambio anual de población de México entre 2006 y 2023 es de aproximadamente **1,329,347.61** personas por año.

**2. ¿Cuál es el valor de la mediana?**

La mediana es de **1,364,369.00** personas. Está cerca de la media, lo que indica que, a diferencia del análisis entre países, la distribución del cambio poblacional de México a lo largo de estos años es bastante simétrica (no hay valores extremos que la distorsionen).

**3. ¿Cuál es el valor de la desviación estándar?**

La desviación estándar es de aproximadamente **252,484.07** personas (varianza ≈ 6.37 × 10¹⁰), lo que muestra una variación moderada del crecimiento poblacional año con año.

**4. ¿En qué año se presentó el valor mínimo y en cuál el máximo?**

- El valor **mínimo** (849,098) se presentó en **2021**, probablemente asociado al impacto de la pandemia de COVID-19 sobre la dinámica poblacional.
- El valor **máximo** (1,625,434) se presentó en **2009**, en la parte más alta de la serie antes de la tendencia decreciente que se observa en años posteriores.

## Actividad 4 — Comprobación del entorno

### Propósito

Comprobar que el notebook está utilizando correctamente las herramientas configuradas para la práctica.

In [14]:
import sys

print("Python:", sys.version)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

Python: 3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]
Pandas: 3.0.5
NumPy: 2.5.2


Las versiones mostradas corresponden al entorno virtual (`.venv`) configurado para este curso, en el cual se instalaron las dependencias listadas en `requirements.txt`.

## Resultados

En este notebook se comprobó el funcionamiento del entorno de trabajo (Python, Pandas, NumPy y Jupyter) utilizando un conjunto de datos real de población mundial:

- Se identificó y citó correctamente la fuente de los datos (ONU, procesados por Our World in Data).
- Se cargó el archivo `annual-population-growth.csv` en un DataFrame de Pandas y se filtró a los datos históricos (1951–2023).
- Se exploró la estructura del DataFrame: dimensiones, columnas, tipos de datos, valores faltantes y cantidad de entidades.
- Se construyó un DataFrame nuevo filtrado para México (país que inicia con "M") desde el año 2006, y se calcularon estadísticos descriptivos (media, mediana, varianza, desviación estándar, mínimo y máximo) sobre la variable de cambio anual de población de ese país.
- Se verificaron las versiones de Python, Pandas y NumPy utilizadas en el entorno virtual del curso.

No se realizaron gráficas, conclusiones, análisis de proyecciones ni análisis temporal, ya que estos temas se incorporarán en prácticas posteriores.

In [15]:
df.loc[:,"Entity"]

0        Afghanistan
1        Afghanistan
2        Afghanistan
3        Afghanistan
4        Afghanistan
            ...     
18683       Zimbabwe
18684       Zimbabwe
18685       Zimbabwe
18686       Zimbabwe
18687       Zimbabwe
Name: Entity, Length: 18688, dtype: str

In [16]:
df.loc[30:35]

,Entity,Code,Year,Annual change in population
30,Afghanistan,AFG,1981,-1231727.0
31,Afghanistan,AFG,1982,-946206.0
32,Afghanistan,AFG,1983,-73395.0
33,Afghanistan,AFG,1984,272237.0
34,Afghanistan,AFG,1985,236633.0
35,Afghanistan,AFG,1986,-6781.0


In [17]:
df.loc[10000]

Entity                          Macao
Code                              MAC
Year                             2023
Annual change in population    9556.0
Name: 10000, dtype: object

In [18]:
df.loc[:,["Entity", "Year", "Annual change in population"]]

,Entity,Year,Annual change in population
0,Afghanistan,1951,103163.0
1,Afghanistan,1952,108441.0
2,Afghanistan,1953,108919.0
3,Afghanistan,1954,111251.0
4,Afghanistan,1955,119027.0
...,...,...,...
18683,Zimbabwe,2019,236920.0
18684,Zimbabwe,2020,255510.0
18685,Zimbabwe,2021,270333.0
18686,Zimbabwe,2022,271841.0


In [19]:
df.loc[10000,["Entity", "Year", "Annual change in population"]]

Entity                          Macao
Year                             2023
Annual change in population    9556.0
Name: 10000, dtype: object

In [20]:
df.loc[(df["Entity"] == "Mexico") & (df["Year"] == 2020)]

,Entity,Code,Year,Annual change in population
10873,Mexico,MEX,2020,1036075.0


In [21]:
df.iloc[:5,0]

0    Afghanistan
1    Afghanistan
2    Afghanistan
3    Afghanistan
4    Afghanistan
Name: Entity, dtype: str

In [22]:
df.iloc[:,[0,2,3]]

,Entity,Year,Annual change in population
0,Afghanistan,1951,103163.0
1,Afghanistan,1952,108441.0
2,Afghanistan,1953,108919.0
3,Afghanistan,1954,111251.0
4,Afghanistan,1955,119027.0
...,...,...,...
18683,Zimbabwe,2019,236920.0
18684,Zimbabwe,2020,255510.0
18685,Zimbabwe,2021,270333.0
18686,Zimbabwe,2022,271841.0
